#### This notebook showcases how to build, run, and evaluate different **Flotorch ADK Agents** with SDK tracing and LLM-based evaluation.
- Sets up a FlotorchADK agent configured on Flotorch Console
- Create tools and connects everything with the agent.
- Sets up Tracing along with the agent
- Use the trace to evaluate the agent's response on different metrics


In [ ]:
!pip install pandas jinja2

In [1]:
import os
import time
from typing import List
from google.adk.runners import Runner
from flotorch.adk.agent import FlotorchADKAgent
from flotorch.adk.sessions import FlotorchADKSession
from google.genai import types
from google.adk.tools import FunctionTool

# Evaluation
from flotorch_eval.agent_eval.metrics.llm_evaluators import (
    TrajectoryEvalWithLLM, 
    TrajectoryEvalWithLLMWithReference, 
    ToolCallAccuracy, 
    AgentGoalAccuracy
    )
from flotorch_eval.agent_eval.metrics.usage_metrics import UsageMetric
from flotorch_eval.agent_eval.metrics.latency_metrics import LatencyMetric
from flotorch_eval.agent_eval.core.client import FlotorchEvalClient
from evaluation_utils import display_evaluation_results


##### Set up the Flotorch API key and base URL
##### This is the only configuration you need to do as it will be used for the agents, tracing and evaluation.


In [ ]:
FLOTORCH_GATEWAY_BASE_URL = "https://dev-gateway.flotorch.cloud"
FLOTORCH_API_KEY = "sk_XKtu651TxvB8C9/asbeXoi9DbuMKWlnSooh1Yt5sjr0=_YTBhZDdkNzYtNGZiZi00MzM4LThiZmQtZDFhNzE5NDZjNDNk_ZTkwN2RkYjEtNWYxYy00Y2ZiLTg3ZjktMWRlMTYyNGYyMmIw"
MEMORY_PROVIDER = "mem0"

### Creating a Runner Factory and a function to run the runner with a query

##### FlotorchADKAgent can be set up using the API key and base url. It can build the agent and the session it requires.

In [ ]:
def create_runner(agent_name: str, tools: List[FunctionTool], enable_sdk_tracing: bool, app_name: str):
    """Factory to create runner for SDK/ADK tracing modes."""
    os.environ["FLOTORCH_ENABLE_SDK_TRACING"] = str(enable_sdk_tracing).lower()

    agent_client = FlotorchADKAgent(
        agent_name=agent_name,
        enable_memory=True,
        custom_tools=tools,
        base_url=FLOTORCH_GATEWAY_BASE_URL,
        api_key=FLOTORCH_API_KEY,
        tracer_config={
            "enabled": True, 
            "sampling_rate": 1
        }
    )
    agent = agent_client.get_agent()

    session_service = FlotorchADKSession(
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_GATEWAY_BASE_URL,
    )

    memory_service = FlotorchMemoryService(
        name=MEMORY_PROVIDER,
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_GATEWAY_BASE_URL,
    )

    return Runner(agent=agent, app_name=app_name, session_service=session_service, memory_service=memory_service), agent_client

# Single Turn Query
def run_single_turn(runner: Runner, query: str, session_id: str, user_id: str) -> str:
    """Send query to agent and return final response text."""
    content = types.Content(role="user", parts=[types.Part(text=query)])
    events = runner.run(user_id=user_id, session_id=session_id, new_message=content)

    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text
    return "No response from agent."

## Evaluation  

- The `FlotorchEvalClient` supports the following metrics:  
  - **TrajectoryEvalWithLLM**  
  - **TrajectoryEvalWithLLMWithReference**
  - **ToolCallAccuracy**  
  - **AgentGoalAccuracy**  
  - **UsageMetric**  
  - **LatencyMetric**  

- `FlotorchEvalClient` only requires an **API key** and **base URL**.  
- Evaluations can be triggered by simply providing a **trace ID**.  
- By default, all available metrics are evaluated.  
- **TrajectoryEvalWithLLMWithReference** is evaluated only if reference is provided.
- A default evaluator can be set on the client, which is used for all metrics that require an LLM.  
- To run specific metrics — or to use a particular model for a given metric — you can define them explicitly and pass them during evaluation.  

**Example:**  

```python
metrics = [
    TrajectoryEvalWithLLM(llm="flotorch/gpt-4o")
    ]
client.evaluate(trace_id, metrics)
```

In [ ]:
async def evaluate_trajectory(trace_id, reference=None, reference_id=None):
    start_time = time.time()
    
    if reference and reference_id:
        raise ValueError("Provide either 'reference' or 'reference_trace_id', not both.")

    client = FlotorchEvalClient(
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_GATEWAY_BASE_URL,
        default_evaluator="flotorch/haiku-long") # Setting a default evaluator for all metrics that require an LLM.

    traces = client.fetch_traces(trace_id)
    print(f"Traces: {traces}")
    results = await client.evaluate( # Metrics can be optionally provided as a list or a single metric
        trace=traces,
        reference=reference,
        reference_trace_id=reference_id
    )
    
    display_evaluation_results(results)
    print(results.model_dump_json(indent=4))
    end_time = time.time()
    time_taken = round(end_time - start_time, 2)
    print(f"Time taken for evaluation: {time_taken} seconds")

## 1. Text Analysis Agent  
- This is a FlotorchADK agent that has access to a single tool.
- **Goal:** Analyze a sentence using the `sentence_breakdown` tool.  
- **Demo Flow:**  
  1. Run agent with a query.  
  2. Capture trace.  
  3. Evaluate trajectory against a reference.

In [ ]:
# Text Analysis Tool
def sentence_breakdown(sentence: str) -> str:
    """
    Break down a sentence into counts of words, characters, and letters.
    Args:
        sentence (str): The input sentence.
    Returns:
        str: A summary of the breakdown.
    """
    words = sentence.split()
    num_words = len(words)
    num_chars = len(sentence)
    num_letters = sum(c.isalpha() for c in sentence)
    num_digits = sum(c.isdigit() for c in sentence)
    num_spaces = sum(c.isspace() for c in sentence)

    return (
        f"Sentence Breakdown:\n"
        f"- Words: {num_words}\n"
        f"- Characters (including spaces): {num_chars}\n"
        f"- Letters: {num_letters}\n"
        f"- Digits: {num_digits}\n"
        f"- Spaces: {num_spaces}"
    )



#### Here we're executing the text-analyzer-agent using the FlotorchADKAgent

In [ ]:
# Main Async Execution
async def run_agent_in_sdk_mode():
    agent_name = "text-analyzer-agent"  # Name of the agent as in the Flotorch Console
    APP_NAME = "text_analysis_app"
    USER_ID = "text_user_001"
    tools = [FunctionTool(sentence_breakdown)]  # List of tools to be used by the agent
    enable_sdk_tracing = False  # Whether to use SDK tracing or not

    runner, agent_client = create_runner(agent_name, tools, enable_sdk_tracing, APP_NAME)

    session1 = await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID,
    )

    query = "The quick brown fox jumps over 13 lazy dogs."
    response = run_single_turn(runner, query, session1.id, USER_ID)

    completed_session = await runner.session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session1.id
    )
    await runner.memory_service.add_session_to_memory(completed_session)

    print("=== SDK Tracing Mode Response ===")
    print(response)

    return agent_client, response


async def main():
    agent_client, response = await run_agent_in_sdk_mode()
    return agent_client

agent_client = await main()


2025-10-08 11:40:21 - flotorch.sdk.llm - INFO - FlotorchLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-10-08 11:40:21 - flotorch.adk.llm - INFO - FlotorchADKLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-10-08 11:40:21 - flotorch.adk.agent - INFO - FlotorchADKAgent initialized (agent_name=text-analyzer-agent, memory_enabled=False)
2025-10-08 11:40:21 - flotorch.sdk.session - INFO - FlotorchSession initialized (base_url=https://dev-gateway.flotorch.cloud)
2025-10-08 11:40:21 - flotorch.adk.sessions - INFO - FlotorchADKSession initialized (base_url=https://dev-gateway.flotorch.cloud)
2025-10-08 11:40:21 - flotorch.sdk.session - INFO - Session Create (uid: 86eb6a3d-1ffe-4b42-a392-9509e7b9fe75) - {'app_name': 'text_analysis_app', 'user_id': 'text_user_001'}
2025-10-08 11:40:22 - flotorch.sdk.session - INFO - Session Created (uid: 86eb6a3d-1ffe-4b42-a392-9509e7b9fe75) - {'app_na

#### Create a reference trajectory that will be used as a reference for **TrajectoryEvalWithLLMWithReference**

You can create a reference in two ways:

- Manually: Write the reference yourself in the required format.

- Using an existing trace: If you already have a trace ID from a previous run, you can pass it to the evaluate method via the reference_trace_id parameter.

`Note`: You can provide either a manual reference or a trace ID, but not both at the same time.

In [8]:
# Sample reference trajectory

REFERENCE_TRAJECTORY = {
    "input": "What is the weather in London and what is 2+2?",
    "expected_steps": [
        {
            "thought": "The user has two distinct questions. I need to get the weather for 'London' first.",
            "tool_call": {
                "name": "get_weather",
                "arguments": {"city": "London"}
            }
        },
        {
            "thought": "Now that I've handled the weather, I need to solve the mathematical expression '2+2'.",
            "tool_call": {
                "name": "calculator",
                "arguments": {"expression": "2+2"}
            }
        },
        {
            "thought": "I have successfully gathered all the necessary information. I will now synthesize the results into a final, coherent answer for the user.",
            "final_response": "The weather in London is currently sunny, and the result of 2+2 is 4."
        }
    ]
}

from flotorch_eval.agent_eval.core.schemas import ReferenceTrajectory
validated_ref = ReferenceTrajectory(**REFERENCE_TRAJECTORY)
print(validated_ref)


input='What is the weather in London and what is 2+2?' expected_steps=[ReferenceStep(thought="The user has two distinct questions. I need to get the weather for 'London' first.", tool_call=ReferenceToolCall(name='get_weather', arguments={'city': 'London'}), final_response=None), ReferenceStep(thought="Now that I've handled the weather, I need to solve the mathematical expression '2+2'.", tool_call=ReferenceToolCall(name='calculator', arguments={'expression': '2+2'}), final_response=None), ReferenceStep(thought='I have successfully gathered all the necessary information. I will now synthesize the results into a final, coherent answer for the user.', tool_call=None, final_response='The weather in London is currently sunny, and the result of 2+2 is 4.')]


#### Evaluate the trajectory using the trace id and the reference trajectory
- A breakdown of all the evaluation will be displayed where each metrics and it's corresponding score (if relevant) and a justification for the score will be provided.

In [ ]:
async def main():
    trace_ids = agent_client.get_tracer_ids() 
    for trace_id in trace_ids:
        if trace_id:
            print(f"Evaluating trace id: {trace_id}")
            await evaluate_trajectory(
                trace_id=trace_id,
                reference=REFERENCE_TRAJECTORY
            )

await main()  

Traces: {'resourceSpans': [{'resource': {'attributes': [{'key': 'service.version', 'value': {'stringValue': '1.0.0'}}, {'key': 'telemetry.sdk.language', 'value': {'stringValue': 'python'}}, {'key': 'telemetry.sdk.name', 'value': {'stringValue': 'opentelemetry'}}, {'key': 'telemetry.sdk.version', 'value': {'stringValue': '1.36.0'}}, {'key': 'service.name', 'value': {'stringValue': 'flotorch-gateway'}}]}, 'scopeSpans': [{'scope': {'name': 'gcp.vertex.agent'}, 'spans': [{'traceId': '9FIJduwX3c/xDt8bA+YeRw==', 'spanId': 'rteXOehm5oo=', 'name': 'invocation', 'kind': 'SPAN_KIND_INTERNAL', 'startTimeUnixNano': '1758793645831064627', 'endTimeUnixNano': '1758793659078332477', 'attributes': [{'key': 'OrgUid', 'value': {'stringValue': 'a0ad7d76-4fbf-4338-8bfd-d1a71946c43d'}}, {'key': 'WorkspaceUid', 'value': {'stringValue': 'e907ddb1-5f1c-4cfb-87f9-1de1624f22b0'}}, {'key': 'ApiKeyUid', 'value': {'stringValue': '3b7d00d8-0457-4115-96e7-ad11b07e3540'}}], 'status': {}}, {'traceId': '9FIJduwX3c/xDt8b

Metric,Score,Details
latency_summary,0.00,Total Latency (Root Steps): 27242.16 ms Average Root Step Latency: 5448.43 ms Latency Breakdown: - invocation: 13247.27 ms - agent_run [text_analyzer_agent]: 10190.21 ms - call_llm: flotorch/haiku-long:latest: 1972.48 ms - execute_tool sentence_breakdown: 0.15 ms - call_llm: flotorch/haiku-long:latest: 1832.05 ms
trajectory_evaluation,1.00,"- details: The agent's goal was to analyze the given sentence ""The quick brown fox jumps over 13 lazy dogs."" and provide a breakdown of its characteristics, such as the number of words, characters, letters, digits, and spaces. The agent successfully achieved this goal by using the ""sentence_breakdown"" tool to analyze the sentence and provide a detailed breakdown. The final output correctly summarizes the sentence's characteristics, including: - Words: 9 - Characters (including spaces): 44 - Letters: 33 - Digits: 2 - Spaces: 8 The agent's reasoning process was logical and coherent, starting with the system message that instructed it to use the provided tools to analyze the given text. It then invoked the ""sentence_breakdown"" tool, which returned the expected breakdown information. The agent then formatted this information into a clear, readable response. Overall, the agent successfully completed the task of analyzing the sentence and providing a detailed breakdown, demonstrating a sound understanding of the goal and using the appropriate tool to achieve it."
toolcall_accuracy,1.00,"- details: The agent's use of the sentence_breakdown tool was appropriate and necessary to provide the requested analysis. The tool was correctly selected, the parameters were accurate, and the output was relevant and complete. The agent did not need to make any additional tool calls to answer the user's request effectively."
agent_goal_accuracy,1.00,"- details: ""user_goal_summary"": ""The user's goal was to have the agent analyze the given sentence 'The quick brown fox jumps over 13 lazy dogs.' and provide a detailed breakdown of its characteristics, including the number of words, characters, letters, digits, and spaces."", ""agent_perception_summary"": ""The agent correctly understood the user's goal to analyze the given sentence and provide a detailed breakdown using the provided 'sentence_breakdown' tool."", ""execution_path_analysis"": ""The agent's execution path was logical and efficient. It first used the 'sentence_breakdown' tool to generate the detailed breakdown of the sentence, and then presented the results in a clear and structured format."", ""final_outcome_evaluation"": ""The agent's final output fully addresses the user's original request. It provides a complete and accurate breakdown of the sentence, including the number of words, characters, letters, digits, and spaces. The information is presented in a clear and easy-to-understand way, meeting all the requirements of the user's goal."", ""overall_conclusion"": ""The agent successfully interpreted the user's goal, formulated an appropriate plan, executed it effectively, and produced a final result that completely fulfills the original request. The agent's performance is accurate and successful in accomplishing the user's true intent."""
usage_summary,0.00,"- total_cost: 0.000406 - average_cost_per_call: 0.000203 - cost_breakdown: - {'model': 'anthropic.claude-3-haiku-20240307-v1:0', 'input_tokens': 435, 'output_tokens': 64, 'cost': '0.000189'} - {'model': 'anthropic.claude-3-haiku-20240307-v1:0', 'input_tokens': 563, 'output_tokens': 61, 'cost': '0.000217'}"
trajectory_evaluation_with_reference,1.00,"- details: The agent's trajectory is correct and functionally equivalent to the reference. The agent used a logical sequence of steps to analyze the given sentence, first breaking it down using the sentence_breakdown tool, and then providing a detailed summary of the sentence's characteristics. The final outcome matches the expected result, and the process followed is valid and relevant to the task."


{
    "trajectory_id": "9FIJduwX3c/xDt8bA+YeRw==",
    "scores": [
        {
            "name": "latency_summary",
            "score": 0.0,
            "details": {
                "total_latency_ms": 27242.16,
                "average_step_latency_ms": 5448.43,
                "latency_breakdown": [
                    {
                        "step_name": "invocation",
                        "latency_ms": 13247.27
                    },
                    {
                        "step_name": "agent_run [text_analyzer_agent]",
                        "latency_ms": 10190.21
                    },
                    {
                        "step_name": "call_llm: flotorch/haiku-long:latest",
                        "latency_ms": 1972.48
                    },
                    {
                        "step_name": "execute_tool sentence_breakdown",
                        "latency_ms": 0.15
                    },
                    {
                        "step_name": "c

## 2. Weather Report Simulator  
- This is a FlotorchADK agent that has access to multiple tools.
- **Goal:** Generate a weather report for a given city.  
- **Tools Used:**  
  - `get_temperature` → Random temperature.  
  - `get_conditions` → Random weather condition.  
  - `generate_advice` → Suggests advice based on condition.  
- **Demo Flow:**  
  1. Agent queries tools in sequence.  
  2. Response combines temperature + condition + advice.  
  3. Evaluated against reference trajectory.

In [ ]:
import random

def get_temperature(city: str) -> int:
    """Return a random temperature in Celsius for the given city.
    
    Args:
        city (str): The name of the city.

    Returns:
        int: A random temperature in Celsius.
    """
    return random.randint(-5, 40)

def get_conditions(city: str) -> str:
    """Return a random weather condition for the given city.
    
    Args:
        city (str): The name of the city.

    Returns:
        str: A random weather condition.
    """
    conditions = ["sunny", "rainy", "cloudy", "stormy", "snowy"]
    return random.choice(conditions)

def generate_advice(condition: str) -> str:
    """Return advice based on the weather condition.
    
    Args:
        condition (str): The weather condition.

    Returns:
        str: Advice based on the weather condition.
    """
    if condition == "rainy":
        return "Carry an umbrella with you."
    elif condition == "sunny":
        return "Wear sunscreen and stay hydrated."
    elif condition == "cloudy":
        return "A light jacket might be useful."
    elif condition == "stormy":
        return "Stay indoors if possible and be cautious."
    elif condition == "snowy":
        return "Dress warmly and watch for icy roads."
    return "Have a great day!"


In [ ]:

# Main Async Execution
async def run_agent_in_sdk_mode():
    APP_NAME = "weather_report_app"
    USER_ID = "weather_user_001"
    agent_name = "weather-report-simulator"
    tools = [FunctionTool(get_temperature), FunctionTool(get_conditions), FunctionTool(generate_advice)]
    enable_sdk_tracing = True

    runner, agent_client = create_runner(agent_name, tools, enable_sdk_tracing, APP_NAME)
    session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

    query = "Generate today’s weather report for Paris"
    response = run_single_turn(runner, query, session.id, USER_ID)

    print("=== SDK Tracing Mode Response ===")
    print(response)
    return agent_client, response

async def main():
    agent_client, response = await run_agent_in_sdk_mode()
    return agent_client

agent_client = await main()  

2025-09-26 11:30:12 - opentelemetry.trace - WARNING - Overriding of current TracerProvider is not allowed
2025-09-26 11:30:13 - flotorch.sdk.llm - INFO - FlotorchLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-09-26 11:30:13 - flotorch.adk.llm - INFO - FlotorchADKLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-09-26 11:30:13 - flotorch.adk.agent - INFO - FlotorchADKAgent initialized (agent_name=weather-report-simulator, memory_enabled=False)
2025-09-26 11:30:13 - flotorch.adk.sessions - INFO - FlotorchADKSession initialized (base_url=https://dev-gateway.flotorch.cloud)
===================START TRACING=====================
🔧 Agent: weather_report_simulator [TRACE_ID: 8a6568c122fd146176661c0a02e3d26d]
🔧 LLM: flotorch/haiku-long:latest [SPAN_ID: a9f60737b10053d9]
🔧 Tool: get_temperature [SPAN_ID: 7c8227bf69836bba]
🔧 LLM: flotorch/haiku-long:latest [SPAN_ID: 41f981cf48b53b66]
🔧 T

In [10]:
REFERENCE_TRAJECTORY = {
    "context": "What is the weather report for Paris?",
    "goal": "Generate a weather report for the requested city, including temperature, condition, and advice.",
    "examples": [
        {
            "thought": "The user wants a weather report for Paris. I need to get the temperature, the weather condition, and then generate advice based on the condition. I will call the tools in this order: `get_temperature`, `get_conditions`, and `generate_advice`.",
            "actions": [
                {
                    "tool_name": "get_temperature",
                    "parameters": {
                        "city": "Paris"
                    }
                },
                {
                    "tool_name": "get_conditions",
                    "parameters": {
                        "city": "Paris"
                    }
                }
            ],
            "response": "null"
        },
        {
            "thought": "I have the temperature (15°C) and the condition ('rainy') for Paris. Now I need to get the appropriate advice for a rainy day by calling the `generate_advice` tool.",
            "actions": [
                {
                    "tool_name": "generate_advice",
                    "parameters": {
                        "condition": "rainy"
                    }
                }
            ],
            "response": "null"
        },
        {
            "thought": "I have successfully gathered all the necessary information: the temperature in Paris is 15°C, the condition is rainy, and the advice is to carry an umbrella. Now I will combine this information into a comprehensive weather report for the user.",
            "actions": [],
            "response": "The weather in Paris is currently 15°C and rainy. Carry an umbrella with you."
        }
    ]
}


In [ ]:
async def main():
    trace_ids = agent_client.get_tracer_ids() 
    for trace_id in trace_ids:
        if trace_id:
            print(f"Evaluating trace id: {trace_id}")
            await evaluate_trajectory(
                trace_id=trace_id,
                reference=REFERENCE_TRAJECTORY
            )

await main()  

Traces: {'resourceSpans': [{'resource': {'attributes': [{'key': 'service.version', 'value': {'stringValue': '1.0.0'}}, {'key': 'telemetry.sdk.language', 'value': {'stringValue': 'python'}}, {'key': 'telemetry.sdk.name', 'value': {'stringValue': 'opentelemetry'}}, {'key': 'telemetry.sdk.version', 'value': {'stringValue': '1.36.0'}}, {'key': 'service.name', 'value': {'stringValue': 'flotorch-gateway'}}]}, 'scopeSpans': [{'scope': {'name': 'gcp.vertex.agent'}, 'spans': [{'traceId': 'r/0U71UpSDI1A1M/QA+e0A==', 'spanId': 'bnUb8cy6G24=', 'name': 'invocation', 'kind': 'SPAN_KIND_INTERNAL', 'startTimeUnixNano': '1758705517667395954', 'endTimeUnixNano': '1758705539892689900', 'attributes': [{'key': 'OrgUid', 'value': {'stringValue': 'a0ad7d76-4fbf-4338-8bfd-d1a71946c43d'}}, {'key': 'WorkspaceUid', 'value': {'stringValue': 'e907ddb1-5f1c-4cfb-87f9-1de1624f22b0'}}, {'key': 'ApiKeyUid', 'value': {'stringValue': '3b7d00d8-0457-4115-96e7-ad11b07e3540'}}], 'status': {}}, {'traceId': 'r/0U71UpSDI1A1M/

Metric,Score,Details
latency_summary,0.00,Total Latency (Root Steps): 50093.67 ms Average Root Step Latency: 4174.47 ms Latency Breakdown: - invocation: 22225.29 ms - agent_run [weather_report_simulator]: 19577.45 ms - call_llm: flotorch/haiku-long:latest: 2750.27 ms - execute_tool get_temperature: 3.04 ms - execute_tool get_temperature: 0.24 ms - call_llm: flotorch/haiku-long:latest: 1805.86 ms - execute_tool get_conditions: 1.34 ms - execute_tool get_conditions: 0.42 ms - call_llm: flotorch/haiku-long:latest: 1842.36 ms - execute_tool generate_advice: 0.57 ms - execute_tool generate_advice: 0.16 ms - call_llm: flotorch/haiku-long:latest: 1886.67 ms
trajectory_evaluation,1.00,"- details: The agent's goal was to generate a complete weather report for Paris, including temperature, weather conditions, and advice. The agent successfully achieved this goal by: 1. Calling the get_temperature tool to retrieve the temperature for Paris (15 degrees Celsius). 2. Calling the get_conditions tool to retrieve the weather conditions for Paris (snowy). 3. Calling the generate_advice tool to generate appropriate advice based on the snowy conditions (dress warmly and watch for icy roads). The agent combined these individual tool outputs into a coherent, natural language weather report. The final report is factually correct and logically complete for the given goal. The agent's reasoning process was clear and efficient, with no redundant or nonsensical steps."
toolcall_accuracy,1.00,"- details: The agent's decision-making in this trajectory was accurate and appropriate. Path A analysis: The agent made three tool calls - get_temperature, get_conditions, and generate_advice. All of these tool calls were necessary and correctly executed to gather the required information to generate a comprehensive weather report for Paris. The temperature, condition, and advice were all relevant and logically combined into the final weather report provided to the user. The tool calls were made in the correct sequence, first gathering the temperature and condition data, then using that to generate the appropriate advice. Overall, the agent demonstrated strong decision-making by utilizing the available tools effectively to produce a high-quality, informative weather report in response to the user's request. No improvements are needed."
agent_goal_accuracy,1.00,"- details: { ""user_goal_summary"": ""The user's goal is to generate a complete weather report for the city of Paris, including the temperature, weather conditions, and advice for the user based on the conditions."", ""agent_perception_summary"": ""The agent correctly perceived its task as generating a full weather report for Paris by using the available tools to gather the necessary information (temperature, conditions, advice) and combining the results into a final report."", ""execution_path_analysis"": ""The agent's execution path was logical and efficient. It followed the prescribed steps of calling the 'get_temperature', 'get_conditions', and 'generate_advice' tools in sequence to gather all the required information. The agent did not get sidetracked or perform any unnecessary actions."", ""final_outcome_evaluation"": ""The agent's final output fully satisfies the user's goal. The weather report includes the temperature, weather conditions, and appropriate advice, all presented in a clear and natural language format."", ""overall_conclusion"": ""The agent successfully interpreted the user's intent, formulated a sound plan, executed it effectively, and produced a complete and accurate weather report for Paris. This aligns perfectly with the user's original request, so the agent's performance is scored as Accurate (1)."" }"
usage_summary,0.00,"- total_cost: 0.001056 - average_cost_per_call: 0.000264 - cost_breakdown: - {'model': 'anthropic.claude-3-haiku-20240307-v1:0', 'input_tokens': 685, 'output_tokens': 53, 'cost': '0.000237'} - {'model': 'anthropic.claude-3-haiku-20240307-v1:0', 'input_tokens': 758,

{
    "trajectory_id": "r/0U71UpSDI1A1M/QA+e0A==",
    "scores": [
        {
            "name": "latency_summary",
            "score": 0.0,
            "details": {
                "total_latency_ms": 50093.67,
                "average_step_latency_ms": 4174.47,
                "latency_breakdown": [
                    {
                        "step_name": "invocation",
                        "latency_ms": 22225.29
                    },
                    {
                        "step_name": "agent_run [weather_report_simulator]",
                        "latency_ms": 19577.45
                    },
                    {
                        "step_name": "call_llm: flotorch/haiku-long:latest",
                        "latency_ms": 2750.27
                    },
                    {
                        "step_name": "execute_tool get_temperature",
                        "latency_ms": 3.04
                    },
                    {
                        "step_name": 

## 3. Fun Fact Generator
- This is an agent that do not have access to any tools. If the agent is still able to answer the query without using any tools, it can be considered a successful run.
- **Goal:** Respond with a fun fact.  
- **Tools:** None (LLM-only reasoning).  
- **Demo Flow:**  
  1. Agent generates fun fact.  
  2. Evaluated without reference trajectory.

In [ ]:

# Main Async Execution
async def run_agent_in_sdk_mode():
    agent_name = "fun-fact-generator"
    APP_NAME = "fun_fact_app"
    USER_ID = "fun_fact_user_001"
    tools = []
    enable_sdk_tracing = True

    runner, agent_client = create_runner(agent_name, tools, enable_sdk_tracing, APP_NAME)
    session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

    query = "Tell me a fun fact."
    response = run_single_turn(runner, query, session.id, USER_ID)

    print("=== SDK Tracing Mode Response ===")
    print(response)
    return agent_client, response

async def main():
    agent_client, response = await run_agent_in_sdk_mode()
    return agent_client

agent_client = await main()

2025-09-18 17:38:00 - opentelemetry.trace - WARNING - Overriding of current TracerProvider is not allowed
2025-09-18 17:38:01 - flotorch.sdk.llm - INFO - FlotorchLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-09-18 17:38:01 - flotorch.adk.llm - INFO - FlotorchADKLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)
2025-09-18 17:38:01 - flotorch.adk.agent - INFO - FlotorchADKAgent initialized (agent_name=fun-fact-generator, memory_enabled=False)
2025-09-18 17:38:01 - flotorch.adk.sessions - INFO - FlotorchADKSession initialized (base_url=https://dev-gateway.flotorch.cloud)
===================START TRACING=====================
🔧 Agent: fun_fact_generator [TRACE_ID: c8ea559ed1a8926413335dcc350d4793]
🔧 LLM: flotorch/haiku-long:latest [SPAN_ID: 9c3e356414ec29b0]
🔧 LLM: flotorch/haiku-long:latest [SPAN_ID: 9c3e356414ec29b0]
🔧 Agent completed [TRACE_ID: c8ea559ed1a8926413335dcc350d4793]=== S

In [ ]:
async def main():
    trace_ids = agent_client.get_tracer_ids() 
    for trace_id in trace_ids:
        if trace_id:
            print(f"Evaluating trace id: {trace_id}")
            await evaluate_trajectory(
                trace_id=trace_id
            )

await main()  

Traces: {'resourceSpans': [{'resource': {'attributes': [{'key': 'service.version', 'value': {'stringValue': '1.0.0'}}, {'key': 'telemetry.sdk.language', 'value': {'stringValue': 'python'}}, {'key': 'telemetry.sdk.name', 'value': {'stringValue': 'opentelemetry'}}, {'key': 'telemetry.sdk.version', 'value': {'stringValue': '1.36.0'}}, {'key': 'service.name', 'value': {'stringValue': 'flotorch-gateway'}}]}, 'scopeSpans': [{'scope': {'name': 'gcp.vertex.agent'}, 'spans': [{'traceId': 'o+TFdih2n3FRyCS9dn1kDA==', 'spanId': 'WZ1MyWOqKO0=', 'parentSpanId': '7f10vF770Qg=', 'name': 'call_llm: flotorch/haiku-long:latest', 'kind': 'SPAN_KIND_INTERNAL', 'startTimeUnixNano': '1758184996452382768', 'endTimeUnixNano': '1758184997694265791', 'attributes': [{'key': 'gen_ai.operation.name', 'value': {'stringValue': 'chat'}}, {'key': 'gen_ai.request.model', 'value': {'stringValue': 'flotorch/haiku-long:latest'}}, {'key': 'gen_ai.request.messages_count', 'value': {'intValue': '2'}}, {'key': 'gen_ai.output.t

Metric,Score,Details
latency_summary,0.00,Total Latency (Root Steps): 9371.05 ms Average Root Step Latency: 3123.68 ms Latency Breakdown: - invocation: 4821.97 ms - agent_run [fun_fact_generator]: 3307.2 ms - call_llm: flotorch/haiku-long:latest: 1241.88 ms
trajectory_evaluation,1.00,"- details: The agent's goal was to generate a single fun fact for the user, as specified in the system message. The agent successfully achieved this goal by responding with the factual statement that ""The first product to have a barcode was Wrigley's gum."" This answer is correct and logically follows from the user's request for a fun fact. The agent's reasoning process was coherent, as it simply generated a short, engaging fact without using any additional tools. Overall, the agent completed the task of providing a fun fact in a clear and appropriate manner."
toolcall_accuracy,1.00,"- details: The agent's decision to not use any tools and provide a direct response with a fun fact was appropriate and accurate in this context. The agent demonstrated the ability to generate a relevant and engaging fun fact from its existing knowledge, without the need for external tools. The provided response is clear, concise, and aligns well with the user's request for a fun fact. Overall, the agent's decision-making process and the final output are evaluated as accurate and effective."
agent_goal_accuracy,1.00,"- details: { ""user_goal_summary"": ""The user's goal is to receive a single, short and factual fun fact from the agent."", ""agent_perception_summary"": ""The agent correctly understood its task to be generating a single fun fact for the user, as per the provided system description."", ""execution_path_analysis"": ""The agent followed a straightforward and efficient path to accomplish its goal. It generated a relevant and concise fun fact about the first product to have a barcode, which aligns with the user's request."", ""final_outcome_evaluation"": ""The agent's final output fully satisfies the user's goal. The provided fun fact is short, factual, and directly addresses the user's request."", ""overall_conclusion"": ""The agent accurately interpreted the user's intent, formulated an appropriate plan, and executed it effectively to deliver a complete and correct response. The agent's performance is deemed successful in fulfilling the user's true goal."" }"
usage_summary,0.00,"- total_cost: 0.000056 - average_cost_per_call: 0.000056 - cost_breakdown: - {'model': 'anthropic.claude-3-haiku-20240307-v1:0', 'input_tokens': 85, 'output_tokens': 28, 'cost': '0.000056'}"


{
    "trajectory_id": "o+TFdih2n3FRyCS9dn1kDA==",
    "scores": [
        {
            "name": "latency_summary",
            "score": 0.0,
            "details": {
                "total_latency_ms": 9371.05,
                "average_step_latency_ms": 3123.68,
                "latency_breakdown": [
                    {
                        "step_name": "invocation",
                        "latency_ms": 4821.97
                    },
                    {
                        "step_name": "agent_run [fun_fact_generator]",
                        "latency_ms": 3307.2
                    },
                    {
                        "step_name": "call_llm: flotorch/haiku-long:latest",
                        "latency_ms": 1241.88
                    }
                ]
            }
        },
        {
            "name": "trajectory_evaluation",
            "score": 1.0,
            "details": {
                "details": "The agent's goal was to generate a single fun fact